## 1. Setup & Imports

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
import torchvision.transforms as transforms
from torchvision.models.video import r3d_18, R3D_18_Weights
import cv2
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 2. Dataset Class

Custom dataset that loads videos and their labels.

In [ ]:
class RouletteVideoDataset(Dataset):
    """
    Dataset for roulette videos.
    Loads video clips and returns tensor of shape (C, T, H, W)
    where C=3 (RGB), T=num_frames, H=height, W=width
    """
    
    def __init__(self, csv_path, dataset_root, num_frames=16, frame_size=112, transform=None):
        """
        Args:
            csv_path: Path to CSV with columns ['input_path', 'label']
            dataset_root: Root directory of the dataset
            num_frames: Number of frames to sample from each video
            frame_size: Size to resize frames to (frame_size x frame_size)
            transform: Optional transforms to apply
        """
        self.df = pd.read_csv(csv_path)
        # Remove rows with missing labels
        self.df = self.df.dropna(subset=['label']).reset_index(drop=True)
        self.dataset_root = Path(dataset_root)
        self.num_frames = num_frames
        self.frame_size = frame_size
        self.transform = transform
        
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        # Get video path and label
        video_rel_path = self.df.iloc[idx]['input_path']
        label = int(self.df.iloc[idx]['label'])
        video_path = self.dataset_root / video_rel_path
        
        # Load video frames
        frames = self._load_video(video_path)
        
        # Convert to tensor (C, T, H, W)
        if self.transform:
            frames = self.transform(frames)
        
        return frames, label
    
    def _load_video(self, video_path):
        """Load video and sample num_frames uniformly"""
        cap = cv2.VideoCapture(str(video_path))
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        
        # Sample frame indices uniformly
        if total_frames < self.num_frames:
            # If video has fewer frames, repeat last frame
            indices = list(range(total_frames)) + [total_frames-1] * (self.num_frames - total_frames)
        else:
            indices = np.linspace(0, total_frames-1, self.num_frames, dtype=int)
        
        frames = []
        for idx in indices:
            cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
            ret, frame = cap.read()
            if ret:
                # Convert BGR to RGB
                frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                # Resize
                frame = cv2.resize(frame, (self.frame_size, self.frame_size))
                # Normalize to [0, 1]
                frame = frame.astype(np.float32) / 255.0
                frames.append(frame)
        
        cap.release()
        
        # Stack frames: (T, H, W, C)
        frames = np.stack(frames, axis=0)
        
        # Convert to (C, T, H, W) for PyTorch
        frames = torch.from_numpy(frames).permute(3, 0, 1, 2)
        
        return frames

# Test the dataset class
print("Dataset class defined successfully!")

## 3. Model Architecture

Using ResNet3D (R3D-18) pre-trained on Kinetics-400.
We'll replace the final classification layer to output 37 classes.

In [ ]:
class RoulettePredictor(nn.Module):
    """
    Video classification model for roulette prediction.
    Based on ResNet3D (R3D-18) with pre-trained weights.
    """
    
    def __init__(self, num_classes=37, freeze_backbone=True):
        """
        Args:
            num_classes: Number of roulette numbers (0-36, so 37 classes)
            freeze_backbone: If True, only train the final layer
        """
        super(RoulettePredictor, self).__init__()
        
        # Load pre-trained R3D-18
        weights = R3D_18_Weights.KINETICS400_V1
        self.model = r3d_18(weights=weights)
        
        # Freeze backbone if specified
        if freeze_backbone:
            for param in self.model.parameters():
                param.requires_grad = False
        
        # Replace final FC layer
        # R3D-18 has fc layer with in_features=512
        in_features = self.model.fc.in_features
        self.model.fc = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(in_features, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )
        
    def forward(self, x):
        """
            x: (batch, C, T, H, W)
        Returns:
            Logits of shape (batch, num_classes)
        """
        return self.model(x)
    
    def unfreeze_backbone(self):
        """Unfreeze all layers for fine-tuning"""
        for param in self.model.parameters():
            param.requires_grad = True

# Test model creation
model = RoulettePredictor(num_classes=37, freeze_backbone=True)
print(f"✅ Model created successfully!")
print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

## 4. Training Setup

Define loss function, optimizer, and training loop.

In [ ]:
def train_epoch(model, dataloader, criterion, optimizer, device):
    """Train for one epoch"""
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    pbar = tqdm(dataloader, desc='Training')
    for videos, labels in pbar:
        videos, labels = videos.to(device), labels.to(device)
        
        # Forward pass
        optimizer.zero_grad()
        outputs = model(videos)
        loss = criterion(outputs, labels)
        
        # Backward pass
        loss.backward()
        optimizer.step()
        
        # Statistics
        running_loss += loss.item() * videos.size(0)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
        
        # Update progress bar
        pbar.set_postfix({'loss': loss.item(), 'acc': 100 * correct / total})
    
    epoch_loss = running_loss / total
    epoch_acc = 100 * correct / total
    return epoch_loss, epoch_acc

def validate_epoch(model, dataloader, criterion, device):
    """Validate for one epoch"""
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        pbar = tqdm(dataloader, desc='Validation')
        for videos, labels in pbar:
            videos, labels = videos.to(device), labels.to(device)
            
            # Forward pass
            outputs = model(videos)
            loss = criterion(outputs, labels)
            
            # Statistics
            running_loss += loss.item() * videos.size(0)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            
            # Update progress bar
            pbar.set_postfix({'loss': loss.item(), 'acc': 100 * correct / total})
    
    epoch_loss = running_loss / total
    epoch_acc = 100 * correct / total
    return epoch_loss, epoch_acc, all_preds, all_labels

print("Training functions defined!")

## 5. Load Dataset & Create DataLoaders

**NOTE**: Before running this, you need:
1. A CSV file with columns: `input_path`, `label`
2. Labels should be integers from 0-36

Update the paths below once you have the labeled data.

In [ ]:
# Configuration
CONFIG = {
    'dataset_root': 'RouletteVision-Dataset',
    'csv_path': 'dataset_labels.csv',  # Update this with your labeled CSV
    'num_frames': 16,
    'frame_size': 112,
    'batch_size': 8,
    'num_workers': 4,
    'train_split': 0.8,
    'learning_rate': 1e-3,
    'num_epochs': 20,
    'save_path': 'roulette_model.pth'
}

# Create dataset (only when you have labels)
# Uncomment when ready:
# dataset = RouletteVideoDataset(
#     csv_path=CONFIG['csv_path'],
#     dataset_root=CONFIG['dataset_root'],
#     num_frames=CONFIG['num_frames'],
#     frame_size=CONFIG['frame_size']
# )
# 
# # Split into train/val
# train_size = int(CONFIG['train_split'] * len(dataset))
# val_size = len(dataset) - train_size
# train_dataset, val_dataset = random_split(dataset, [train_size, val_size])
# 
# # Create dataloaders
# train_loader = DataLoader(
#     train_dataset, 
#     batch_size=CONFIG['batch_size'], 
#     shuffle=True, 
#     num_workers=CONFIG['num_workers'],
#     pin_memory=True
# )
# 
# val_loader = DataLoader(
#     val_dataset, 
#     batch_size=CONFIG['batch_size'], 
#     shuffle=False, 
#     num_workers=CONFIG['num_workers'],
#     pin_memory=True
# )
# 
# print(f"✅ Dataset loaded: {len(dataset)} videos")
# print(f"   Training: {train_size} videos")
# print(f"   Validation: {val_size} videos")

print("Configuration set. Uncomment code above when labels are ready.")

## 6. Training Loop

Main training loop - run this to train the model.

In [ ]:
# Training function
def train_model(model, train_loader, val_loader, config, device):
    """Full training loop"""
    
    # Setup
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=config['learning_rate']
    )
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=3, verbose=True
    )
    
    # Move model to device
    model = model.to(device)
    
    # Training history
    history = {
        'train_loss': [], 'train_acc': [],
        'val_loss': [], 'val_acc': []
    }
    
    best_val_acc = 0.0
    
    print(f"\n{'='*60}")
    print(f"Starting training for {config['num_epochs']} epochs")
    print(f"{'='*60}\n")
    
    for epoch in range(config['num_epochs']):
        print(f"\nEpoch {epoch+1}/{config['num_epochs']}")
        print("-" * 60)
        
        # Train
        train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
        
        # Validate
        val_loss, val_acc, _, _ = validate_epoch(model, val_loader, criterion, device)
        
        # Learning rate scheduling
        scheduler.step(val_loss)
        
        # Save history
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        
        # Print epoch summary
        print(f"\nEpoch {epoch+1} Summary:")
        print(f"  Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}%")
        print(f"  Val Loss:   {val_loss:.4f} | Val Acc:   {val_acc:.2f}%")
        
        # Save best model
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), config['save_path'])
            print(f"  ✅ New best model saved! (Val Acc: {val_acc:.2f}%)")
    
    print(f"\n{'='*60}")
    print(f"Training completed!")
    print(f"Best validation accuracy: {best_val_acc:.2f}%")
    print(f"Model saved to: {config['save_path']}")
    print(f"{'='*60}\n")
    
    return history

# Uncomment to train when data is ready:
# history = train_model(model, train_loader, val_loader, CONFIG, device)

print("Training function ready. Uncomment above when data is loaded.")

## 7. Visualization & Evaluation

Plot training curves and evaluate model performance.

In [ ]:
def plot_training_history(history):
    """Plot training and validation metrics"""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
    
    epochs = range(1, len(history['train_loss']) + 1)
    
    # Loss plot
    ax1.plot(epochs, history['train_loss'], 'b-', label='Train Loss')
    ax1.plot(epochs, history['val_loss'], 'r-', label='Val Loss')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Loss')
    ax1.set_title('Training and Validation Loss')
    ax1.legend()
    ax1.grid(True)
    
    # Accuracy plot
    ax2.plot(epochs, history['train_acc'], 'b-', label='Train Acc')
    ax2.plot(epochs, history['val_acc'], 'r-', label='Val Acc')
    ax2.axhline(y=100/37, color='gray', linestyle='--', label='Random (2.7%)')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Accuracy (%)')
    ax2.set_title('Training and Validation Accuracy')
    ax2.legend()
    ax2.grid(True)
    
    plt.tight_layout()
    plt.show()

def evaluate_model(model, val_loader, device):
    """Detailed model evaluation"""
    model.eval()
    criterion = nn.CrossEntropyLoss()
    
    val_loss, val_acc, all_preds, all_labels = validate_epoch(
        model, val_loader, criterion, device
    )
    
    print(f"\n{'='*60}")
    print("Final Evaluation Results")
    print(f"{'='*60}")
    print(f"Validation Loss: {val_loss:.4f}")
    print(f"Validation Accuracy: {val_acc:.2f}%")
    print(f"Random Baseline: {100/37:.2f}%")
    print(f"Improvement: {val_acc / (100/37):.2f}x over random")
    
    # Classification report
    print("\nClassification Report:")
    print(classification_report(all_labels, all_preds, target_names=[str(i) for i in range(37)]))
    
    # Confusion matrix
    cm = confusion_matrix(all_labels, all_preds)
    plt.figure(figsize=(12, 10))
    sns.heatmap(cm, annot=False, fmt='d', cmap='Blues')
    plt.title('Confusion Matrix')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.show()
    
    return val_acc

# Uncomment after training:
# plot_training_history(history)
# final_acc = evaluate_model(model, val_loader, device)

print("Evaluation functions ready!")

## 8. Inference Function

Make predictions on new videos.

In [ ]:
def predict_video(model, video_path, dataset_root, num_frames=16, frame_size=112, device='cuda'):
    """
    Predict roulette outcome for a single video
    
    Returns:
        predicted_number: Most likely number (0-36)
        probabilities: Probability distribution over all 37 numbers
    """
    model.eval()
    
    # Create temporary dataset for single video
    # (In practice, you'd load the video directly)
    temp_df = pd.DataFrame([{
        'input_path': video_path,
        'label': 0  # Dummy label
    }])
    temp_csv = 'temp_inference.csv'
    temp_df.to_csv(temp_csv, index=False)
    
    dataset = RouletteVideoDataset(
        csv_path=temp_csv,
        dataset_root=dataset_root,
        num_frames=num_frames,
        frame_size=frame_size
    )
    
    video_tensor, _ = dataset[0]
    video_tensor = video_tensor.unsqueeze(0).to(device)  # Add batch dimension
    
    with torch.no_grad():
        outputs = model(video_tensor)
        probabilities = torch.softmax(outputs, dim=1)[0]
        predicted_number = torch.argmax(probabilities).item()
    
    # Clean up
    Path(temp_csv).unlink()
    
    return predicted_number, probabilities.cpu().numpy()

# Example usage (uncomment after training):
# video_path = 'Input-Output Videos/SET 1/S1_INPUT_1.mp4'
# pred_num, probs = predict_video(model, video_path, CONFIG['dataset_root'], device=device)
# print(f"Predicted number: {pred_num}")
# print(f"Confidence: {probs[pred_num]*100:.2f}%")
# 
# # Show top 5 predictions
# top5_indices = np.argsort(probs)[-5:][::-1]
# print("\nTop 5 predictions:")
# for idx in top5_indices:
#     print(f"  Number {idx}: {probs[idx]*100:.2f}%")

print("Inference function ready!")

## 9. Summary & Next Steps

### What we have:
- ✅ Dataset class for loading videos
- ✅ Pre-trained ResNet3D model with custom classification head
- ✅ Training and validation loops
- ✅ Evaluation and visualization functions
- ✅ Inference function for new videos

### To start training:
1. **Get labels**: Create `dataset_labels.csv` with columns `input_path` and `label` (0-36)
2. **Uncomment data loading** in cell 6
3. **Run training** by uncommenting cell 7
4. **Evaluate** using cell 8

### Expected timeline:
- Data preparation: 2-3 hours
- Training (20 epochs): 4-6 hours
- Evaluation & demo prep: 2-3 hours

### Expected performance:
- Random baseline: ~2.7% accuracy
- Target: 8-15% accuracy (3-5x better than random)
- Good result: >30% top-5 accuracy